# Resource Cleanup

In this notebook, we'll clean up the AWS resources we created during the workshop to avoid incurring unnecessary charges.

## 1. Import Dependencies

In [ ]:
import os
import json
import boto3
import time

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Provide opportunity to set values here if needed
    print("\nYou can also set the values directly here:")
    new_bucket = input("S3 Bucket Name: ")
    if new_bucket:
        S3_BUCKET = new_bucket
        
    new_region = input("AWS Region: ")
    if new_region:
        AWS_REGION = new_region
        
    new_role = input("SageMaker Role ARN: ")
    if new_role:
        SAGEMAKER_ROLE_ARN = new_role
        
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN

## 3. Load Deployment Information

In [ ]:
# Load deployment information if available
try:
    with open('deployment_info.json', 'r') as f:
        deployment_info = json.load(f)
    print("Loaded deployment information")
except FileNotFoundError:
    deployment_info = {}
    print("No deployment information found")

## 3. Delete SageMaker Endpoints

In [ ]:
# Create SageMaker client
sagemaker_client = boto3.client('sagemaker', region_name=AWS_REGION)

# Delete endpoints
for model_type, info in deployment_info.items():
    if "endpoint_name" in info:
        endpoint_name = info["endpoint_name"]
        print(f"Deleting endpoint {endpoint_name}...")
        try:
            sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
            print(f"Endpoint {endpoint_name} deleted successfully")
        except Exception as e:
            print(f"Error deleting endpoint {endpoint_name}: {e}")

## 4. Delete S3 Objects

In [ ]:
# Create S3 client
s3_client = boto3.client('s3', region_name=AWS_REGION)

# List objects in the bucket
print(f"Listing objects in bucket {S3_BUCKET}...")
response = s3_client.list_objects_v2(Bucket=S3_BUCKET)

# Delete objects
if 'Contents' in response:
    for obj in response['Contents']:
        key = obj['Key']
        print(f"Deleting object {key}...")
        s3_client.delete_object(Bucket=S3_BUCKET, Key=key)
        print(f"Object {key} deleted successfully")
else:
    print("No objects found in the bucket")

## 5. Delete S3 Bucket

In [ ]:
# Delete the bucket
print(f"Deleting bucket {S3_BUCKET}...")
try:
    s3_client.delete_bucket(Bucket=S3_BUCKET)
    print(f"Bucket {S3_BUCKET} deleted successfully")
except Exception as e:
    print(f"Error deleting bucket {S3_BUCKET}: {e}")

## 6. Delete IAM Role

In [ ]:
# Note: We don't delete the SageMaker execution role since it's managed by SageMaker
print("Skipping IAM role deletion as we're using the SageMaker execution role")

## 7. Clean Up Local Files (Optional)

In [ ]:
# This cell is commented out by default to avoid accidentally deleting files
# Uncomment and run if you want to clean up local files

'''
import shutil

# Delete models directory
if os.path.exists("models"):
    print("Deleting models directory...")
    shutil.rmtree("models")
    print("Models directory deleted successfully")

# Delete JSON files
json_files = [
    'model_info.json',
    'baseline_metrics.json',
    'quantized_metrics.json',
    'pruned_metrics.json',
    'deployment_info.json'
]

for file in json_files:
    if os.path.exists(file):
        print(f"Deleting {file}...")
        os.remove(file)
        print(f"{file} deleted successfully")
'''

## 8. Verify Cleanup

In [ ]:
# Check if endpoints still exist
print("Checking for SageMaker endpoints...")
try:
    response = sagemaker_client.list_endpoints()
    endpoints = [endpoint['EndpointName'] for endpoint in response['Endpoints']]
    
    workshop_endpoints = []
    for model_type, info in deployment_info.items():
        if "endpoint_name" in info and info["endpoint_name"] in endpoints:
            workshop_endpoints.append(info["endpoint_name"])
    
    if workshop_endpoints:
        print(f"Warning: The following workshop endpoints still exist: {workshop_endpoints}")
    else:
        print("All workshop endpoints have been deleted successfully")
except Exception as e:
    print(f"Error checking endpoints: {e}")

# Check if bucket still exists
print(f"\nChecking if bucket {S3_BUCKET} still exists...")
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"Warning: Bucket {S3_BUCKET} still exists")
except:
    print(f"Bucket {S3_BUCKET} has been deleted successfully")

## 9. Workshop Completion

Congratulations! You've completed the Model Optimization Workshop and cleaned up all the resources you created.

### What You've Learned

1. How to download and prepare models from Hugging Face
2. How to establish baseline performance metrics
3. How to apply quantization techniques to reduce model size and improve inference speed
4. How to apply pruning techniques to further optimize models
5. How to deploy models to AWS SageMaker
6. How to compare performance and cost between baseline and optimized models
7. How to clean up AWS resources

### Next Steps

- Apply these optimization techniques to your own models
- Explore more advanced optimization techniques like knowledge distillation
- Experiment with different pruning ratios and quantization methods
- Consider combining multiple optimization techniques for even better results

Thank you for participating in this workshop!